# CNN Reproduction Tests — Attention Is Not All You Need for Diffraction

Reproduces **Table I**, **Fig. S1**, **Table S2**, **Table S8**, and **Table S9**.

Set `DATA_DIR` in the **Configuration** cell, then run sections top-to-bottom.

In [ ]:
import sys
import warnings
from collections import OrderedDict
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path('.').resolve()))
from resnet_model import ResnetClassifier, ModelOutput
from dataset import get_dataloaders

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Architecture & Helpers

In [ ]:
# Confirmed architecture (matches all checkpoints; strides verified against training configs)
ARCH_EG = dict(
    input_dim=1,
    res_dims=[32, 64, 64, 64],
    res_kernel=[5, 7, 17, 13],
    res_stride=[4, 4, 5, 3],
    num_blocks=[2, 2, 2, 2],
    first_kernel_size=13,
    first_stride=1,
    first_pool_kernel_size=7,
    first_pool_stride=7,
    num_classes=99,
    block_type='basic',
)
ARCH_SPG = {**ARCH_EG, 'num_classes': 230}  # space-group variant


def load_model(checkpoint_path, arch=ARCH_EG):
    m = ResnetClassifier(
        input_dim=arch['input_dim'],
        res_dims=arch['res_dims'],
        res_kernel=arch['res_kernel'],
        res_stride=arch['res_stride'],
        num_blocks=arch['num_blocks'],
        first_kernel_size=arch['first_kernel_size'],
        first_stride=arch['first_stride'],
        first_pool_kernel_size=arch['first_pool_kernel_size'],
        first_pool_stride=arch['first_pool_stride'],
        num_classes=arch['num_classes'],
        block_type=arch['block_type'],
    )
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    sd = ckpt.get('model', ckpt)
    sd = OrderedDict((k.replace('module.', ''), v) for k, v in sd.items())
    m.load_state_dict(sd, strict=False)
    return m.to(device).eval()


def get_test_loader(h5_path, num_classes, batch_size=128):
    _, _, loader, _, _ = get_dataloaders(
        h5_file=str(h5_path),
        batch_size=batch_size,
        num_classes=num_classes,
        num_workers=0,
        prefetch_factor=None,
    )
    return loader


def evaluate(model, loader):
    top1 = top3 = top5 = total = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out = ModelOutput(model(X))
            n = len(y)
            total += n
            top1 += out.top_k_acc(y, 1) * n / 100
            top3 += out.top_k_acc(y, 3) * n / 100
            top5 += out.top_k_acc(y, 5) * n / 100
    return {
        'top1': round(top1 / total * 100, 2),
        'top3': round(top3 / total * 100, 2),
        'top5': round(top5 / total * 100, 2),
    }

## Configuration
Set `DATA_DIR` to the folder containing all HDF5 dataset files, then run the sanity-check cell below.

In [ ]:
CKPT_DIR = Path('checkpoints')

# ── SET THIS to the folder containing all HDF5 files ────────────────────────
DATA_DIR = Path('/path/to/data')
# ────────────────────────────────────────────────────────────────────────────

BATCH_SIZE = 128

# ── Dataset files (filenames as provided; just update DATA_DIR above) ────────

# Table I — SG vs EG
SPG_H5  = DATA_DIR / '1000_ref_spg_2_3m_highres.hdf5'   # SPG model train/eval data
EXT_H5  = DATA_DIR / '1000_ref_ext_990k_highres.hdf5'   # EG model (990k) train/eval data

# Fig. S1 — scaling curve (each file contains the matching train-size split)
SCALING_H5 = {
      '99k': DATA_DIR / '1000_ref_ext_99k_highres.hdf5',
     '396k': DATA_DIR / '1000_ref_ext_396k_highres.hdf5',
     '792k': DATA_DIR / '1000_ref_ext_792k_highres.hdf5',
     '990k': DATA_DIR / '1000_ref_ext_990k_highres.hdf5',
    '1.48M': DATA_DIR / '1000_ref_ext_1_48m_highres.hdf5',
    '2.17M': DATA_DIR / '1000_ref_ext_2_17m_highres.hdf5',
    '3.16M': DATA_DIR / '1000_ref_ext_3_16m_highres.hdf5',
    '3.76M': DATA_DIR / '1000_ref_ext_3_76m_highres.hdf5',
    '4.15M': DATA_DIR / '1000_ref_ext_4_15m_highres.hdf5',
    '5.14M': DATA_DIR / '1000_ref_ext_5_14m_highres.hdf5',
}

# Table S2 / S8 — biased vs balanced cross-eval
BALANCED_H5  = DATA_DIR / '10million_pyxtal_repacked_cleaned.hdf5'  # unbiased / balanced
ICSD_BIAS_H5 = DATA_DIR / '10M_W_ICSD_Bias_pyxtal.hdf5'            # ICSD-induced bias
AUGMENTED_H5 = DATA_DIR / 'rruff_aug_10M_trainready.hdf5'           # augmented (Table S8)

# Table S9 — ICSD de-duplication
NON_DEDUP_H5 = DATA_DIR / 'Ext_ICSD_NonDeDup_1000_hr.h5'
DEDUP_H5     = DATA_DIR / 'Ext_ICSD_DeDup_1000_hr.h5'

# ── Sanity check ─────────────────────────────────────────────────────────────
all_files = (
    [('Table I',  SPG_H5), ('Table I', EXT_H5)]
    + [('Fig. S1', p) for p in SCALING_H5.values()]
    + [('Table S2/S8', BALANCED_H5), ('Table S2/S8', ICSD_BIAS_H5), ('Table S8', AUGMENTED_H5)]
    + [('Table S9', NON_DEDUP_H5), ('Table S9', DEDUP_H5)]
)
for section, p in all_files:
    print(f"{'OK' if p.exists() else 'MISSING':7s}  [{section}]  {p.name}")

---
## Table I — Space-Group vs Extinction-Group ResNet-18

| Model | Checkpoint | Dataset |
|---|---|---|
| SPG ResNet-18 (230 classes) | `xrd_model_ve988dop.pth` | `1000_ref_spg_2_3m_highres.hdf5` |
| EG ResNet-18  (99 classes)  | `xrd_model_t75aaxpz.pth` | `1000_ref_ext_990k_highres.hdf5` |

In [ ]:
spg_loader = get_test_loader(SPG_H5, num_classes=230, batch_size=BATCH_SIZE)
eg_loader  = get_test_loader(EXT_H5, num_classes=99,  batch_size=BATCH_SIZE)

spg_model = load_model(CKPT_DIR / 'xrd_model_ve988dop.pth', arch=ARCH_SPG)
eg_model  = load_model(CKPT_DIR / 'xrd_model_t75aaxpz.pth', arch=ARCH_EG)

spg_res = evaluate(spg_model, spg_loader)
eg_res  = evaluate(eg_model,  eg_loader)

df_table1 = pd.DataFrame([
    {'Model': 'SPG ResNet-18 (230 classes)', **spg_res},
    {'Model': 'EG ResNet-18  (99 classes)',  **eg_res},
])
df_table1.columns = ['Model', 'Top-1 (%)', 'Top-3 (%)', 'Top-5 (%)']
display(df_table1)

---
## Fig. S1 — ResNet-18 Scaling Curve

Each model was trained on a different-size subset of the extinction-group dataset.
Evaluated on the test split of its own matched HDF5 file.

| Training size | Checkpoint | Dataset |
|---|---|---|
| 99k   | `xrd_model_d7bovz1p.pth` | `1000_ref_ext_99k_highres.hdf5` |
| 396k  | `xrd_model_9ytnffm7.pth` | `1000_ref_ext_396k_highres.hdf5` |
| 792k  | `xrd_model_q3hxsqw8.pth` | `1000_ref_ext_792k_highres.hdf5` |
| 990k  | `xrd_model_t75aaxpz.pth` | `1000_ref_ext_990k_highres.hdf5` |
| 1.48M | `xrd_model_wj5yemvc.pth` | `1000_ref_ext_1_48m_highres.hdf5` |
| 2.17M | `xrd_model_fmwj0hcj.pth` | `1000_ref_ext_2_17m_highres.hdf5` |
| 3.16M | `xrd_model_ujr9w21e.pth` | `1000_ref_ext_3_16m_highres.hdf5` |
| 3.76M | `xrd_model_eiyfd7gf.pth` | `1000_ref_ext_3_76m_highres.hdf5` |
| 4.15M | `xrd_model_srq8yg04.pth` | `1000_ref_ext_4_15m_highres.hdf5` |
| 5.14M | `xrd_model_jek05aky.pth` | `1000_ref_ext_5_14m_highres.hdf5` |

In [ ]:
SCALING_MODELS = [
    ('99k',   'xrd_model_d7bovz1p.pth'),
    ('396k',  'xrd_model_9ytnffm7.pth'),
    ('792k',  'xrd_model_q3hxsqw8.pth'),
    ('990k',  'xrd_model_t75aaxpz.pth'),
    ('1.48M', 'xrd_model_wj5yemvc.pth'),
    ('2.17M', 'xrd_model_fmwj0hcj.pth'),
    ('3.16M', 'xrd_model_ujr9w21e.pth'),
    ('3.76M', 'xrd_model_eiyfd7gf.pth'),
    ('4.15M', 'xrd_model_srq8yg04.pth'),
    ('5.14M', 'xrd_model_jek05aky.pth'),
]

rows_s1 = []
for size_label, ckpt_name in SCALING_MODELS:
    loader = get_test_loader(SCALING_H5[size_label], num_classes=99, batch_size=BATCH_SIZE)
    model  = load_model(CKPT_DIR / ckpt_name, arch=ARCH_EG)
    res    = evaluate(model, loader)
    rows_s1.append({'Training size': size_label, **res})
    print(f'{size_label:>6}  top1={res["top1"]:.2f}%  top3={res["top3"]:.2f}%  top5={res["top5"]:.2f}%')

df_s1 = pd.DataFrame(rows_s1)

# Numeric x-axis in millions
size_to_m = {'99k': 0.099, '396k': 0.396, '792k': 0.792, '990k': 0.990,
             '1.48M': 1.48, '2.17M': 2.17, '3.16M': 3.16,
             '3.76M': 3.76, '4.15M': 4.15, '5.14M': 5.14}
x = [size_to_m[s] for s in df_s1['Training size']]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x, df_s1['top1'], 'o-', label='Top-1')
ax.plot(x, df_s1['top3'], 's-', label='Top-3')
ax.plot(x, df_s1['top5'], '^-', label='Top-5')
ax.set_xlabel('Training set size (millions)')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Fig. S1 — ResNet-18 Scaling Curve')
ax.legend()
plt.tight_layout()
plt.show()

display(df_s1)

---
## Table S2 — Biased vs Balanced ResNet-18 Cross-Evaluation

Each model is evaluated on **both** test distributions (cross-eval design).

| Training distribution | Checkpoint | Training dataset |
|---|---|---|
| Unbiased / Balanced | `xrd_model_ivvi2z1j.pth` | `10million_pyxtal_repacked_cleaned.hdf5` |
| ICSD-induced bias   | `xrd_model_m9mdi5p9.pth` | `10M_W_ICSD_Bias_pyxtal.hdf5` |

---
## Table S8 — Biased / Balanced / Augmented Cross-Evaluation

Extends Table S2 with a third model trained on augmented data.

| Training distribution | Checkpoint | Training dataset |
|---|---|---|
| Unbiased / Balanced | `xrd_model_ivvi2z1j.pth` | `10million_pyxtal_repacked_cleaned.hdf5` |
| ICSD-induced bias   | `xrd_model_m9mdi5p9.pth` | `10M_W_ICSD_Bias_pyxtal.hdf5` |
| Augmented           | `xrd_model_kiifq20z.pth` | `rruff_aug_10M_trainready.hdf5` |

In [ ]:
# Each model is evaluated on all three test sets (cross-eval)
CROSSEVAL_MODELS = [
    # (label,                    checkpoint,                      Table scope)
    ('Balanced   (ivvi2z1j)', CKPT_DIR / 'xrd_model_ivvi2z1j.pth'),  # Table S2 + S8
    ('ICSD Bias  (m9mdi5p9)', CKPT_DIR / 'xrd_model_m9mdi5p9.pth'),  # Table S2 + S8
    ('Augmented  (kiifq20z)', CKPT_DIR / 'xrd_model_kiifq20z.pth'),  # Table S8 only
]

TEST_SETS = [
    ('Balanced test',  BALANCED_H5,  99),
    ('ICSD Bias test', ICSD_BIAS_H5, 99),
    ('Augmented test', AUGMENTED_H5, 99),
]

rows_cross = []
for model_label, ckpt_path in CROSSEVAL_MODELS:
    m = load_model(ckpt_path, arch=ARCH_EG)
    for test_label, h5_path, n_cls in TEST_SETS:
        loader = get_test_loader(h5_path, num_classes=n_cls, batch_size=BATCH_SIZE)
        res = evaluate(m, loader)
        rows_cross.append({
            'Model (train dist.)': model_label,
            'Test set': test_label,
            'Top-1 (%)': res['top1'],
            'Top-3 (%)': res['top3'],
            'Top-5 (%)': res['top5'],
        })

df_cross = pd.DataFrame(rows_cross)
print('=== Table S2 (Balanced vs ICSD Bias rows only) ===')
display(df_cross[df_cross['Model (train dist.)'].str.contains('ivvi2z1j|m9mdi5p9') &
                 df_cross['Test set'].isin(['Balanced test', 'ICSD Bias test'])])
print('\n=== Table S8 (all three models, all three test sets) ===')
display(df_cross)

---
## Table S9 — Effect of ICSD De-duplication

Each model is evaluated on **both** test sets (cross-eval design).

| Condition | Checkpoint | Training dataset |
|---|---|---|
| Non-deduplicated | `xrd_model_u9quo4zn.pth` | `Ext_ICSD_NonDeDup_1000_hr.h5` |
| Deduplicated     | `xrd_model_xhx3ex54.pth` | `Ext_ICSD_DeDup_1000_hr.h5` |

In [ ]:
DEDUP_MODELS = [
    ('Non-deduplicated (u9quo4zn)', CKPT_DIR / 'xrd_model_u9quo4zn.pth'),
    ('Deduplicated     (xhx3ex54)', CKPT_DIR / 'xrd_model_xhx3ex54.pth'),
]

DEDUP_TEST_SETS = [
    ('Non-dedup test', NON_DEDUP_H5, 99),
    ('Dedup test',     DEDUP_H5,     99),
]

rows_dup = []
for model_label, ckpt_path in DEDUP_MODELS:
    m = load_model(ckpt_path, arch=ARCH_EG)
    for test_label, h5_path, n_cls in DEDUP_TEST_SETS:
        loader = get_test_loader(h5_path, num_classes=n_cls, batch_size=BATCH_SIZE)
        res = evaluate(m, loader)
        rows_dup.append({
            'Model (train condition)': model_label,
            'Test set': test_label,
            'Top-1 (%)': res['top1'],
            'Top-3 (%)': res['top3'],
            'Top-5 (%)': res['top5'],
        })

display(pd.DataFrame(rows_dup))